# Lesson 2.5c — Part III：RAG 检索联调

对比四种 chunk 来源的问答效果。

- **Embedding**：本地 `fastembed` + `all-MiniLM-L6-v2`（ONNX，无需 transformers）
- **LLM**：DeepSeek API（需 `DEEPSEEK_API_KEY`）

上一步：[05a_part1_pipeline.ipynb](05a_part1_pipeline.ipynb) / [05b_part2_advanced_parsing.ipynb](05b_part2_advanced_parsing.ipynb)

---


---
# Part III — RAG 检索联调（pdf_parsing2_n）

对比四种 chunk 来源的问答效果：
1. **Pipeline JSONL**（tiktoken + 页码元数据）— 推荐
2. **Table Markdown**（`03` 产出，`out/markdown/*.md`，每张表一个 chunk）
3. **Marker Markdown**（`05b` 产出，按 `#` 标题切分，选修）
4. **LlamaParse**（按页，选修）

需要 `DEEPSEEK_API_KEY`；Embedding 本地运行，无需 OpenAI Key。

```mermaid
flowchart LR
    A[all_chunks] --> B[chunks_to_langchain_docs]
    C[marker.md] --> D[marker_to_docs<br/>按 # 标题切]
    E[llamaparse_pages] --> F[llamaparse_to_docs<br/>按页]
    B --> G[FastEmbed<br/>all-MiniLM-L6-v2]
    D --> G
    F --> G
    G --> H[FAISS 向量库<br/>k=5 检索]
    H --> I[ConversationalRetrievalChain]
    I --> J[DeepSeek 问答<br/>引用 page / chunk_id]

    style A fill:#1565C0,color:#fff
    style B fill:#1565C0,color:#fff
    style C fill:#E65100,color:#fff
    style D fill:#E65100,color:#fff
    style E fill:#6A1B9A,color:#fff
    style F fill:#6A1B9A,color:#fff
    style G fill:#00695C,color:#fff
    style H fill:#00838F,color:#fff
    style I fill:#4527A0,color:#fff
    style J fill:#2E7D32,color:#fff
```

## 0. 安装依赖


In [1]:
%pip install langchain-openai langchain-community langchain-classic faiss-cpu python-dotenv fastembed -q


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import re
from dataclasses import dataclass
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(override=True)


def _resolve_here() -> Path:
    """定位 05_PDF_process 目录（兼容 kernel cwd 为项目根或本目录）。"""
    for base in (Path.cwd(), Path.cwd() / "02-RAG" / "05_PDF_process"):
        if (base / "doc" / "attention_is_all_you_need.pdf").exists():
            return base.resolve()
    return Path.cwd().resolve()


HERE = _resolve_here()
PDF_PATH = HERE / "doc" / "attention_is_all_you_need.pdf"
CHUNKS_JSONL = HERE / "out" / "chunks" / f"{PDF_PATH.stem}_chunks.jsonl"
TABLE_MD_DIR = HERE / "out" / "markdown"
MARKER_MD = HERE / "out" / f"{PDF_PATH.stem}_marker.md"
LLAMAPARSE_JSON = HERE / "out" / f"{PDF_PATH.stem}_llamaparse.json"

print(f"工作目录: {HERE}")
print(f"chunks 文件: {CHUNKS_JSONL}")


@dataclass
class Chunk:
    file: str
    page: int
    chunk_id: int
    text: str
    token_count: int
    source_type: str


def load_chunks(jsonl_path: Path) -> list[Chunk]:
    if not jsonl_path.exists():
        raise FileNotFoundError(f"请先运行 05a 生成: {jsonl_path}")
    chunks = []
    for line in jsonl_path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            chunks.append(Chunk(**json.loads(line)))
    return chunks


def load_table_markdown_files(md_dir: Path) -> list[Path]:
    if not md_dir.exists():
        return []
    return sorted(md_dir.glob("*.md"))


all_chunks = load_chunks(CHUNKS_JSONL)
table_md_files = load_table_markdown_files(TABLE_MD_DIR)
marker_text = MARKER_MD.read_text(encoding="utf-8") if MARKER_MD.exists() else ""
llamaparse_pages: list[str] = (
    json.loads(LLAMAPARSE_JSON.read_text(encoding="utf-8")) if LLAMAPARSE_JSON.exists() else []
)

print(f"Pipeline chunks: {len(all_chunks)}")
print(f"Table markdown: {len(table_md_files)} 个 ({TABLE_MD_DIR})")
print(f"Marker md: {'有' if marker_text.strip() else '无'} ({MARKER_MD})")
print(f"LlamaParse: {'有' if llamaparse_pages else '无'} ({LLAMAPARSE_JSON})")


工作目录: D:\workspace\doc\面试狂魔\人工智能面试题\AI_coding_interview\02-RAG\05_PDF_process
chunks 文件: D:\workspace\doc\面试狂魔\人工智能面试题\AI_coding_interview\02-RAG\05_PDF_process\out\chunks\attention_is_all_you_need_chunks.jsonl
Pipeline chunks: 46
Table markdown: 8 个 (D:\workspace\doc\面试狂魔\人工智能面试题\AI_coding_interview\02-RAG\05_PDF_process\out\markdown)
Marker md: 无 (D:\workspace\doc\面试狂魔\人工智能面试题\AI_coding_interview\02-RAG\05_PDF_process\out\attention_is_all_you_need_marker.md)
LlamaParse: 无 (D:\workspace\doc\面试狂魔\人工智能面试题\AI_coding_interview\02-RAG\05_PDF_process\out\attention_is_all_you_need_llamaparse.json)


In [3]:
def chunks_to_langchain_docs(chunks: list[Chunk]):
    from langchain_core.documents import Document
    return [
        Document(
            page_content=c.text,
            metadata={"page": c.page, "chunk_id": c.chunk_id, "source_type": c.source_type, "file": c.file},
        )
        for c in chunks
    ]


def table_md_to_docs(md_paths: list[Path]):
    from langchain_core.documents import Document
    docs = []
    for p in md_paths:
        text = p.read_text(encoding="utf-8").strip()
        if not text:
            continue
        meta = {"source_type": "table_md", "file": p.name}
        m = re.match(r"plumber_p(\d+)_t(\d+)", p.stem)
        if m:
            meta["page"] = int(m.group(1))
            meta["table_id"] = int(m.group(2))
        docs.append(Document(page_content=text, metadata=meta))
    return docs


def marker_to_docs(md_text: str):
    from langchain_core.documents import Document
    pattern = re.compile(r"(?m)^# .*?(?=^# |\Z)", re.DOTALL)
    parts = pattern.findall(md_text) if md_text.strip() else []
    return [Document(page_content=p, metadata={"section": i, "source_type": "marker"}) for i, p in enumerate(parts)]


def llamaparse_to_docs(pages: list[str]):
    from langchain_core.documents import Document
    return [Document(page_content=t, metadata={"page": i + 1, "source_type": "llamaparse"}) for i, t in enumerate(pages) if t.strip()]

In [ ]:
from fastembed import TextEmbedding
from langchain_core.embeddings import Embeddings
from langchain_openai import ChatOpenAI


class FastEmbedEmbeddings(Embeddings):
    """FastEmbed ONNX 封装，避免 Windows 下 transformers 长路径问题。"""

    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self._model = TextEmbedding(model_name)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [list(v) for v in self._model.embed(texts)]

    def embed_query(self, text: str) -> list[float]:
        return list(next(self._model.embed([text])))


EMBEDDINGS = FastEmbedEmbeddings()
DEEPSEEK_MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-chat")


def build_rag_chain(docs, model: str | None = None):
    from langchain_community.vectorstores import FAISS
    from langchain_classic.chains.conversational_retrieval.base import ConversationalRetrievalChain
    from langchain_core.prompts import PromptTemplate

    if not docs:
        raise ValueError("文档为空，无法建索引")
    api_key = "sk-your-api-key-here"

    llm = ChatOpenAI(
        model=model or DEEPSEEK_MODEL,
        temperature=0,
        api_key=api_key,
        base_url="https://api.deepseek.com",
    )
    db = FAISS.from_documents(docs, EMBEDDINGS)
    retriever = db.as_retriever(search_kwargs={"k": 5})

    prompt = PromptTemplate(
        input_variables=["question", "context"],
        template=("根据以下 context 回答问题，并引用 page/chunk_id。\n\n"
                  "Context:\n{context}\n\nQuestion:\n{question}"),
    )
    return ConversationalRetrievalChain.from_llm(
        llm, retriever, return_source_documents=True,
        combine_docs_chain_kwargs={"prompt": prompt},
    )


def ask(chain, question: str):
    from pprint import pprint
    resp = chain.invoke({"question": question, "chat_history": []})
    print("Q:", question)
    print()
    print("answer")
    pprint(resp["answer"])
    print("\n来源:")
    for i, d in enumerate(resp.get("source_documents", [])[:3]):
        print(f"  [{i}] meta={d.metadata} | {d.page_content[:120]}...")
    return resp

### 11.1 Pipeline chunks → FAISS → 问答

In [10]:
QUESTION = "文档主要内容是什么？"  # 英文论文可改为 Transformer 相关问题

try:
    pipeline_docs = chunks_to_langchain_docs(all_chunks)
    chain_pipeline = build_rag_chain(pipeline_docs)
    resp_pipeline = ask(chain_pipeline, QUESTION)
except Exception as e:
    print(f"Pipeline RAG 跳过: {e}")

Q: 文档主要内容是什么？
('根据提供的上下文，文档主要内容是关于**注意力机制（attention '
 'heads）在神经网络中的可视化分析**，特别是针对**输入-输入层（Input-Input Layer）** 中不同注意力头的行为和功能。\n'
 '\n'
 '具体来说，文档通过图表（Figure 4 和 Figure 5）展示了：\n'
 '- 在层5（layer 5 of 6）中，两个注意力头（head 5 和 head '
 '6）似乎参与了**照应（anaphora）解析**，并且对特定单词（如“its”）的注意力非常尖锐（sharp）（来源：Figure 4 '
 '描述，chunk_id: 第14页）。\n'
 '- 许多注意力头表现出与**句子结构（structure of the sentence）** '
 '相关的行为，并且不同的头学会了执行不同的任务（来源：Figure 5 描述，chunk_id: 第15页）。\n'
 '\n'
 '此外，上下文开头包含一段反转文本（“The Law will never be perfect, but its application should '
 'be just - this is what we are missing, in my '
 'opinion.”），这可能是作为示例句子用于分析。文档还引用了多篇关于神经机器翻译、语法解析和计算机视觉的学术文献（如[37]至[40]），表明该内容可能来自一篇学术论文或技术报告，主题涉及**神经网络中的注意力机制及其在自然语言处理中的应用**。')

来源:
  [0] meta={'page': 14, 'chunk_id': 41, 'source_type': 'table', 'file': 'attention_is_all_you_need.pdf'} | | ehT | waL | lliw | reven | eb | tcefrep | , | tub | sti | noitacilp | dluohs | eb | tsuj | - | siht | si | tahw | ew |...
  [1] meta={'page': 15, 'chunk_id': 44, 'source_type': 'table', '

### 11.2 Table Markdown → FAISS → 问答（`03` 产出）

In [11]:
if table_md_files:
    try:
        chain_table = build_rag_chain(table_md_to_docs(table_md_files))
        resp_table = ask(chain_table, QUESTION)
    except Exception as e:
        print(f"Table Markdown RAG 跳过: {e}")
else:
    print("无 Table Markdown，请先运行 03_extract_tables.ipynb")

Q: 文档主要内容是什么？
('根据提供的上下文，文档主要内容是关于**注意力机制（attention heads）** 和**模型训练配置**的描述。具体包括：\n'
 '\n'
 '- 文档提到了“注意力头”（attention heads）的分布和隔离，例如“Two attention heads, also in layer 5 '
 'of 6, apparently involved in anaphora resolution”（第4行，chunk_id: 4）和“Isolate '
 'attention heads from just the word ‘its’ for attention heads '
 '5-6”（第5行，chunk_id: 5）。\n'
 '- 文档还包含一个**训练配置表**，列出了模型参数（如层数、隐藏维度、注意力头数等）和训练方式（如“WSJ only, '
 'discriminative”、“semi-supervised”、“multi-task”、“generative”）（第7-8行，chunk_id: '
 '7-8）。\n'
 '\n'
 '因此，文档主要内容是**关于注意力机制的分析以及模型训练的超参数和策略**。')

来源:
  [0] meta={'source_type': 'table_md', 'file': 'plumber_p14_t1.md', 'page': 14, 'table_id': 1} | | ehT   | waL   | lliw   | reven   | eb   | tcefrep   | ,   | tub   | sti   | noitacilp   | dluohs   | eb.1   | tsuj   |...
  [1] meta={'source_type': 'table_md', 'file': 'plumber_p15_t1.md', 'page': 15, 'table_id': 1} | | ehT   |   Unnamed: 1 | waL   |   Unnamed: 3 | lliw   |   Unnamed: 5 | reven   |   Unnamed: 7 | eb   |   Unnamed: 9 | t...
  

### 11.3 Marker Markdown → FAISS → 问答（`05b` 产出，选修）

In [7]:
if marker_text.strip():
    try:
        chain_marker = build_rag_chain(marker_to_docs(marker_text))
        resp_marker = ask(chain_marker, QUESTION)
    except Exception as e:
        print(f"Marker RAG 跳过: {e}")
else:
    print("无 Marker 输出，跳过（需运行 05b）")

无 Marker 输出，跳过（需运行 05b）


### 11.4 LlamaParse 按页 → FAISS → 问答

In [8]:
if llamaparse_pages:
    try:
        chain_llama = build_rag_chain(llamaparse_to_docs(llamaparse_pages))
        resp_llama = ask(chain_llama, QUESTION)
    except Exception as e:
        print(f"LlamaParse RAG 跳过: {e}")
else:
    print("无 LlamaParse 输出，跳过")

无 LlamaParse 输出，跳过


## 12. 四种路径对比小结

| 路径 | Chunk 策略 | 元数据 | 成本 | 质量 |
|------|-----------|--------|------|------|
| **Pipeline** | tiktoken + overlap | file/page/chunk_id | 低 | 稳定可控 |
| **Table Markdown** | 每张表一个 md | page/table_id | 低 | 表格问答好 |
| **Marker** | Markdown `#` 标题 | section | 中（本地 GPU） | 复杂版面好 |
| **LlamaParse** | 按页 | page | 高（云 API） | 通常最高 |

> **工程建议**：生产用 Part I Pipeline；表格密集场景用 `03` 的 Table Markdown；质量 benchmark 用 Part II/III 对比后再选型。

### 常见陷阱 Top 10
1. 数字 PDF 跑 OCR  2. 两栏直接拼  3. 大 PDF 一次读  4. 页眉页脚未去
5. 连字符未合并  6. 表格被切碎  7. 按字符切 chunk  8. 无元数据
9. OCR 无中文包  10. 无置信度过滤